In [ ]:
Install all required libraries

In [ ]:

!pip install -q ultralytics opencv-python-headless pandas numpy

print("Installation completed successfully!")


In [ ]:
Upload your traffic video with code also

In [ ]:
from google.colab import files
import cv2
import numpy as np
import pandas as pd
import os

uploaded = files.upload()
video_path = next(iter(uploaded.keys()))

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

duration = total_frames / fps if fps > 0 else 0

print("VIDEO UPLOADED SUCCESSFULLY")
print("Video:", video_path)
print("Resolution:", width, "x", height)
print("FPS:", fps)
print("Duration:", round(duration, 2), "seconds")

cap.release()


In [ ]:
Configure AI model and speed calibration

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLO model
model = YOLO("yolo11n.pt")

# Vehicle classes
VEHICLE_CLASSES = [1, 2, 3, 5, 7]

# Speed thresholds (km/h)
GREEN_SPEED = 30
YELLOW_SPEED = 10

# IMPORTANT SPEED CALIBRATION
# Example only: replace with measured pixels per metre
PIXELS_PER_METER = 8.0

# Smoothing to reduce sudden speed changes
SMOOTHING = 0.3

print("YOLO model loaded successfully!")
print("Vehicle detection enabled!")
print("ByteTrack tracking enabled!")
print("Speed estimation configured!")
print("Remember to calibrate pixels per metre.")


In [ ]:
Main AI traffic monitoring system

In [ ]:
from collections import Counter

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

cap.release()

os.makedirs("traffic_results", exist_ok=True)

output_path = "traffic_results/AI_Traffic_Output.mp4"

writer = cv2.VideoWriter(
    output_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

# Store tracking information
previous_positions = {}
speed_history = {}
vehicle_speeds = {}
vehicle_types = {}

unique_vehicles = set()
speed_records = []
frame_number = 0

print("Starting traffic monitoring...")

results = model.track(
    source=video_path,
    stream=True,
    tracker="bytetrack.yaml",
    classes=VEHICLE_CLASSES,
    conf=0.20,
    persist=True,
    verbose=False
)

for result in results:

    frame = result.orig_img.copy()

    current_ids = set()
    current_speeds = []

    if result.boxes is not None and result.boxes.id is not None:

        boxes = result.boxes.xyxy.cpu().numpy()
        ids = result.boxes.id.int().cpu().tolist()
        classes = result.boxes.cls.int().cpu().tolist()

        for box, track_id, cls in zip(boxes, ids, classes):

            x1, y1, x2, y2 = map(int, box)

            # Vehicle centre
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2

            current_ids.add(track_id)
            unique_vehicles.add(track_id)

            vehicle_name = model.names[cls]
            vehicle_types[track_id] = vehicle_name

            current_time = frame_number / fps

            # Calculate movement between frames
            if track_id in previous_positions:

                old_x, old_y, old_time = previous_positions[track_id]

                elapsed = current_time - old_time

                if elapsed > 0:

                    pixel_distance = np.sqrt(
                        (cx - old_x) ** 2 +
                        (cy - old_y) ** 2
                    )

                    # Convert pixel movement to approximate km/h
                    speed = (
                        pixel_distance /
                        PIXELS_PER_METER /
                        elapsed
                    ) * 3.6

                    # Ignore tracking jitter and unrealistic jumps
                    if pixel_distance >= 1 and speed <= 180:

                        old_speed = speed_history.get(track_id, speed)

                        smooth_speed = (
                            SMOOTHING * speed +
                            (1 - SMOOTHING) * old_speed
                        )

                        speed_history[track_id] = smooth_speed
                        vehicle_speeds[track_id] = smooth_speed

            previous_positions[track_id] = (
                cx, cy, current_time
            )

            # Assign traffic colours
            if track_id not in vehicle_speeds:
                color = (255, 255, 255)
                status = "TRACKING"

            else:
                speed = vehicle_speeds[track_id]
                current_speeds.append(speed)

                if speed >= GREEN_SPEED:
                    color = (0, 255, 0)
                    status = "NORMAL"

                elif speed >= YELLOW_SPEED:
                    color = (0, 255, 255)
                    status = "SLOW"

                else:
                    color = (0, 0, 255)
                    status = "VERY SLOW"

            # Draw coloured vehicle box
            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                color,
                2
            )

            if track_id in vehicle_speeds:

                label = (
                    f"ID:{track_id} {vehicle_name} "
                    f"{vehicle_speeds[track_id]:.1f} km/h "
                    f"{status}"
                )

                speed_records.append({
                    "Vehicle ID": track_id,
                    "Vehicle Type": vehicle_name,
                    "Speed (km/h)": round(
                        vehicle_speeds[track_id], 2
                    )
                })

            else:
                label = f"ID:{track_id} {vehicle_name} TRACKING"

            cv2.putText(
                frame,
                label,
                (x1, max(y1 - 8, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.45,
                color,
                2
            )

    # Current frame vehicle count
    current_count = len(current_ids)

    # Average speed of currently tracked vehicles
    average_speed = (
        np.mean(current_speeds)
        if current_speeds else 0
    )

    # Congestion classification
    if current_count >= 10 and average_speed < 15:
        traffic_status = "HEAVY TRAFFIC"
        traffic_color = (0, 0, 255)

    elif current_count >= 5 or (
        current_speeds and average_speed < 20
    ):
        traffic_status = "MODERATE TRAFFIC"
        traffic_color = (0, 255, 255)

    else:
        traffic_status = "LOW TRAFFIC"
        traffic_color = (0, 255, 0)

    # Dashboard
    cv2.rectangle(frame, (5, 5), (360, 125), (0, 0, 0), -1)

    cv2.putText(
        frame,
        f"Current Vehicles: {current_count}",
        (15, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6, (255, 255, 255), 2
    )

    cv2.putText(
        frame,
        f"Total Unique: {len(unique_vehicles)}",
        (15, 58),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6, (255, 255, 255), 2
    )

    cv2.putText(
        frame,
        f"Average Speed: {average_speed:.1f} km/h",
        (15, 86),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55, (255, 255, 255), 2
    )

    cv2.putText(
        frame,
        traffic_status,
        (15, 113),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65, traffic_color, 2
    )

    writer.write(frame)

    frame_number += 1

    if frame_number % 100 == 0:
        print("Processed frames:", frame_number)

writer.release()

print("\nVIDEO PROCESSING COMPLETED!")
print("Total unique vehicles:", len(unique_vehicles))
print("Vehicles with speed estimates:", len(vehicle_speeds))
print("Output:", output_path)


In [ ]:
Final results, vehicle count and dashboard

In [ ]:
from IPython.display import display, Video
from google.colab import files

# Total unique vehicles
total_count = len(unique_vehicles)

# Vehicle category count
category_counts = Counter(vehicle_types.values())

# Create one speed record per vehicle
final_records = []

for vehicle_id, speed in vehicle_speeds.items():

    final_records.append({
        "Vehicle ID": vehicle_id,
        "Vehicle Type": vehicle_types.get(vehicle_id, "Unknown"),
        "Estimated Speed (km/h)": round(speed, 2)
    })

speed_df = pd.DataFrame(final_records)

# Calculate average speed
if not speed_df.empty:
    average_speed = speed_df["Estimated Speed (km/h)"].mean()
else:
    average_speed = 0

# Final traffic condition
if total_count >= 10 and average_speed < 15:
    final_status = "HEAVY TRAFFIC"
elif total_count >= 5:
    final_status = "MODERATE TRAFFIC"
else:
    final_status = "LOW TRAFFIC"

# Print final report
print("=" * 45)
print("       AI TRAFFIC MONITORING REPORT")
print("=" * 45)

print("\nTOTAL VEHICLES DETECTED:", total_count)

print("\nVEHICLE CATEGORY COUNTS:")
for name, count in category_counts.items():
    print(name, ":", count)

print("\nVEHICLES WITH SPEED ESTIMATES:", len(speed_df))

print("\nAVERAGE VEHICLE SPEED:",
      round(average_speed, 2), "km/h")

print("\nTRAFFIC CONDITION:", final_status)

print("\nINDIVIDUAL VEHICLE SPEED REPORT:")

if not speed_df.empty:
    display(speed_df)
else:
    print("No speed estimates available.")

# Save report
speed_df.to_csv(
    "traffic_results/vehicle_speed_report.csv",
    index=False
)

# Display processed video
print("\nPROCESSED TRAFFIC VIDEO:")

display(Video(output_path, embed=True, width=800))

# Download results
files.download(output_path)
files.download("traffic_results/vehicle_speed_report.csv")

print("\nALL RESULTS GENERATED SUCCESSFULLY!")
